# Step 5F ? Targeted hard-negative CNN retraining for v6

The alpha-0.50 model materially reduced aggregate false turns in fresh live shadow, but a sustained high-confidence forward->right block remained. Step 5E then captured a clean full manual traversal and isolated the exact visual states where the human command was forward while the shadow model predicted right.

This notebook changes the training data only: it adds a deterministic set of 40 clean human-forward hard-negative/context frames to the training split. The architecture, preprocessing, optimizer, random seed, early stopping, validation session, and reference test session remain unchanged. Class-weight alpha stays fixed at 0.50 so the effect of the targeted data can be measured directly.


## Locked session split and hard-negative rule

- Train: original_01, original_02, original_03, recovery_01, recovery_03, hardneg_clean_20261018
- Validation: recovery_02
- Reference test: recovery_04

The hard-negative session contributes training data only. Selection was fixed before v6 training: include every clean stable human=forward/model=right frame plus up to three neighboring chronological frames on either side when those context frames are also stable human=forward. This yields exactly 40 added forward-labeled images, including all 17 clean false-right frames. The interrupted laptop-hang session is excluded.

The validation and reference-test sessions remain unchanged. recovery_04 has already been inspected in earlier experiments, so it is retained as a repeat reference, not described as a pristine untouched final estimate. A fresh live shadow traversal remains the decisive external gate.


In [ ]:
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/Intelligent_Robotics_Design_Project"
SESSION_DATASET_ZIP_NAME = "step05f_hard_negative_dataset_v1.zip"
OUTPUT_MODEL_NAME = "maze_navigation_model_v6.h5"
OUTPUT_METRICS_NAME = "step05f_v6_metrics.json"
OUTPUT_CONFUSION_NAME = "step05f_v6_reference_confusion.csv"
OUTPUT_HISTORY_NAME = "step05f_v6_history.csv"

RANDOM_SEED = 42
IMAGE_SIZE = (50, 50)
BATCH_SIZE = 32
EPOCHS = 30
CLASS_WEIGHT_ALPHA = 0.50

CLASS_NAMES = ("forward", "left", "right")
CLASS_TO_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}

EXPECTED_SPLIT_SESSIONS = {
    "train": {"original_01", "original_02", "original_03", "recovery_01", "recovery_03", "hardneg_clean_20261018"},
    "validation": {"recovery_02"},
    "test": {"recovery_04"},
}


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import shutil
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

tf.keras.utils.set_random_seed(RANDOM_SEED)
tf.config.experimental.enable_op_determinism()

project_dir = Path(DRIVE_PROJECT_DIR)
zip_path = project_dir / SESSION_DATASET_ZIP_NAME
if not zip_path.is_file():
    raise FileNotFoundError(f"Upload the session-aware dataset ZIP here first: {zip_path}")

print(f"TensorFlow: {tf.__version__}")
print(f"Dataset ZIP: {zip_path}")


In [ ]:
extract_root = Path("/content/step05f_hard_negative_dataset")
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)

with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall(extract_root)

manifest_path = extract_root / "manifest.csv"
split_plan_path = extract_root / "split_plan.csv"
if not manifest_path.is_file() or not split_plan_path.is_file():
    raise FileNotFoundError("Dataset package must contain manifest.csv and split_plan.csv")

manifest = pd.read_csv(manifest_path)
required_columns = {"session_id", "split", "label", "relative_path", "source_filename"}
if set(manifest.columns) != required_columns:
    raise ValueError(f"Unexpected manifest columns: {list(manifest.columns)}")

if not set(manifest["label"]).issubset(CLASS_TO_INDEX):
    raise ValueError("Manifest contains an unknown class label")

manifest["label_index"] = manifest["label"].map(CLASS_TO_INDEX).astype(np.int32)
manifest["image_path"] = manifest["relative_path"].map(lambda value: str(extract_root / value))

missing = [path for path in manifest["image_path"] if not Path(path).is_file()]
if missing:
    raise FileNotFoundError(f"Missing extracted images, first example: {missing[0]}")

for split, expected_sessions in EXPECTED_SPLIT_SESSIONS.items():
    actual_sessions = set(manifest.loc[manifest["split"] == split, "session_id"])
    if actual_sessions != expected_sessions:
        raise ValueError(
            f"{split} session mismatch: expected {sorted(expected_sessions)}, got {sorted(actual_sessions)}"
        )

all_session_sets = list(EXPECTED_SPLIT_SESSIONS.values())
for i in range(len(all_session_sets)):
    for j in range(i + 1, len(all_session_sets)):
        if not all_session_sets[i].isdisjoint(all_session_sets[j]):
            raise ValueError("A session appears in more than one split")

print("SESSION ISOLATION CHECK: PASS")
print(f"Total images: {len(manifest)}")


In [ ]:
selection_path = extract_root / "hard_negative_selection.csv"
if not selection_path.is_file():
    raise FileNotFoundError("Dataset package must contain hard_negative_selection.csv")
hard_negative_selection = pd.read_csv(selection_path)
if len(hard_negative_selection) != 40:
    raise ValueError(f"Expected 40 hard-negative/context frames, got {len(hard_negative_selection)}")
print("TARGETED HARD-NEGATIVE SELECTION: PASS")
print(hard_negative_selection["reason"].value_counts().to_dict())
display(hard_negative_selection.head(10))


In [ ]:
summary = (
    manifest.groupby(["split", "session_id", "label"])
    .size()
    .rename("images")
    .reset_index()
)
display(summary)

split_counts = (
    manifest.groupby(["split", "label"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=CLASS_NAMES, fill_value=0)
)
display(split_counts)

for split in ("train", "validation", "test"):
    frame = manifest[manifest["split"] == split]
    print(f"{split}: {len(frame)} images | sessions={sorted(frame['session_id'].unique().tolist())}")


In [ ]:
def load_image(path):
    image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    return tf.cast(image, tf.float32) / 255.0

def load_example(path, label):
    return load_image(path), label

def make_dataset(frame, training=False):
    dataset = tf.data.Dataset.from_tensor_slices(
        (frame["image_path"].to_numpy(), frame["label_index"].to_numpy(dtype=np.int32))
    )
    dataset = dataset.map(load_example, num_parallel_calls=tf.data.AUTOTUNE, deterministic=True)
    if training:
        dataset = dataset.shuffle(len(frame), seed=RANDOM_SEED, reshuffle_each_iteration=True)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_df = manifest[manifest["split"] == "train"].reset_index(drop=True)
val_df = manifest[manifest["split"] == "validation"].reset_index(drop=True)
test_df = manifest[manifest["split"] == "test"].reset_index(drop=True)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)

print("Datasets ready.")


In [ ]:
def build_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(*IMAGE_SIZE, 3)),
        tf.keras.layers.Conv2D(16, (3, 3), activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(3, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

model = build_model()
model.summary()


In [ ]:
train_labels = train_df["label_index"].to_numpy(dtype=np.int32)
train_counts = np.bincount(train_labels, minlength=len(CLASS_NAMES))

balanced = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(CLASS_NAMES)),
    y=train_labels,
).astype(np.float64)

raw_weights = np.power(balanced, CLASS_WEIGHT_ALPHA)
mean_sample_weight = float(np.dot(train_counts, raw_weights) / train_counts.sum())
normalized = raw_weights / mean_sample_weight
class_weights = {index: float(normalized[index]) for index in range(len(CLASS_NAMES))}

print("Fixed class-weight alpha:", CLASS_WEIGHT_ALPHA)
print("Training counts:", {CLASS_NAMES[i]: int(train_counts[i]) for i in range(3)})
print("Class weights:", {CLASS_NAMES[i]: class_weights[i] for i in range(3)})


In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=2,
)

history_df = pd.DataFrame(history.history)
print(f"Epochs completed: {len(history_df)}")
display(history_df.tail())

plt.figure(figsize=(8, 4))
plt.plot(history_df.index + 1, history_df["loss"], label="train loss")
plt.plot(history_df.index + 1, history_df["val_loss"], label="validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


In [ ]:
def prediction_metrics(frame, probabilities):
    labels = frame["label_index"].to_numpy(dtype=np.int32)
    predictions = np.argmax(probabilities, axis=1)
    confusion = confusion_matrix(labels, predictions, labels=np.arange(3))
    precision, recall, f1, support = precision_recall_fscore_support(
        labels, predictions, labels=np.arange(3), zero_division=0
    )
    forward_index = CLASS_TO_INDEX["forward"]
    left_index = CLASS_TO_INDEX["left"]
    right_index = CLASS_TO_INDEX["right"]
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "macro_f1": float(np.mean(f1)),
        "precision": {CLASS_NAMES[i]: float(precision[i]) for i in range(3)},
        "recall": {CLASS_NAMES[i]: float(recall[i]) for i in range(3)},
        "f1": {CLASS_NAMES[i]: float(f1[i]) for i in range(3)},
        "support": {CLASS_NAMES[i]: int(support[i]) for i in range(3)},
        "forward_to_left": int(confusion[forward_index, left_index]),
        "forward_to_right": int(confusion[forward_index, right_index]),
        "forward_support": int(confusion[forward_index].sum()),
        "confusion_matrix": confusion,
    }

val_probabilities = model.predict(val_ds, verbose=0)
val_metrics = prediction_metrics(val_df, val_probabilities)
val_confusion = val_metrics.pop("confusion_matrix")

print(f"Validation accuracy: {val_metrics['accuracy']:.2%}")
print(f"Validation macro-F1: {val_metrics['macro_f1']:.4f}")
print("Validation recall:", {k: f"{v:.2%}" for k, v in val_metrics["recall"].items()})
print(
    "Validation forward->left/right:",
    val_metrics["forward_to_left"],
    val_metrics["forward_to_right"],
    "of",
    val_metrics["forward_support"],
)
display(pd.DataFrame(val_confusion, index=CLASS_NAMES, columns=CLASS_NAMES))


## Held-out test evaluation

The model is now fixed. The recovery_04 test session is evaluated below once, after training and validation are complete.


In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
test_probabilities = model.predict(test_ds, verbose=0)
test_metrics = prediction_metrics(test_df, test_probabilities)
test_confusion = test_metrics.pop("confusion_matrix")

print(f"TEST loss: {test_loss:.4f}")
print(f"TEST accuracy: {test_accuracy:.4f} ({test_accuracy:.2%})")
print(f"TEST macro-F1: {test_metrics['macro_f1']:.4f}")
print("TEST recall:", {k: f"{v:.2%}" for k, v in test_metrics["recall"].items()})
print(
    "TEST forward->left/right:",
    test_metrics["forward_to_left"],
    test_metrics["forward_to_right"],
    "of",
    test_metrics["forward_support"],
)
display(pd.DataFrame(test_confusion, index=CLASS_NAMES, columns=CLASS_NAMES))


In [ ]:
project_dir.mkdir(parents=True, exist_ok=True)

model_path = project_dir / OUTPUT_MODEL_NAME
metrics_path = project_dir / OUTPUT_METRICS_NAME
confusion_path = project_dir / OUTPUT_CONFUSION_NAME
history_path = project_dir / OUTPUT_HISTORY_NAME

model.save(model_path)
pd.DataFrame(test_confusion, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(confusion_path)
history_df.to_csv(history_path, index=False)

metrics_payload = {
    "method": "session-aware targeted hard-negative retraining",
    "random_seed": RANDOM_SEED,
    "image_size": list(IMAGE_SIZE),
    "class_weight_alpha": CLASS_WEIGHT_ALPHA,
    "split_sessions": {key: sorted(value) for key, value in EXPECTED_SPLIT_SESSIONS.items()},
    "train_images": len(train_df),
    "validation_images": len(val_df),
    "test_images": len(test_df),
    "epochs_completed": len(history_df),
    "validation": val_metrics,
    "test": test_metrics,
}
metrics_path.write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")

print(f"Saved v6 model: {model_path}")
print(f"Saved metrics: {metrics_path}")
print(f"Saved test confusion: {confusion_path}")
print(f"Saved training history: {history_path}")


## Step-6 gate after v6

Do not move directly to autonomous motor control from this notebook.

First compare v6 against the alpha-0.50 baseline on recovery_02 and the repeat recovery_04 reference. Then place maze_navigation_model_v6.h5 in the local models folder and run a fresh human-controlled no-display shadow traversal. The key live criteria are forward false-turn rate, longest forward->right streak, and left/right recall. Autonomous Step 6 remains blocked until the fresh sequential shadow evidence is satisfactory.
